# Financial linking and abnormal return extraction

This notebook takes the completed textual-analysis output and the daily stock return data, then:

1. loads and cleans the three input files
2. links each earnings call to a `PERMNO` using `ticker`
3. computes daily abnormal returns as `DlyRet - vwretd`
4. extracts short-run `CAR[-1,+1]` around each earnings call
5. extracts a medium-run CAR[+2,+30] abnormal return (trading days +2 to +30 after the call)
6. merges the financial variables back into the textual-analysis table
7. exports a regression-ready dataset

**Input files expected in the same folder as this notebook**
- `textual_analysis_results.csv`
- `daily_ret.csv`
- `linking_table.csv`

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. File paths

In [2]:
from pathlib import Path

BASE = Path("../../data")

TA_RES_PATH = BASE / "processed/textual_analysis_results.csv"
RET_PATH = BASE / "raw/daily_ret.csv"
LINK_PATH = BASE / "raw/linking_table.csv"

print(TA_RES_PATH.resolve())
print(RET_PATH.resolve())
print(LINK_PATH.resolve())
print("TA exists:", TA_RES_PATH.exists())
print("RET exists:", RET_PATH.exists())
print("LINK exists:", LINK_PATH.exists())

/Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/processed/textual_analysis_results.csv
/Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/raw/daily_ret.csv
/Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/raw/linking_table.csv
TA exists: True
RET exists: True
LINK exists: True


## 2. Load datasets

Note: `textual_analusis_results.csv` is semicolon-separated, and several sentiment/intensity variables use commas as decimal separators.

In [3]:
ta = pd.read_csv(TA_RES_PATH)
ret = pd.read_csv(RET_PATH)
link = pd.read_csv(LINK_PATH)

print("ta shape  :", ta.shape)
print("ret shape :", ret.shape)
print("link shape:", link.shape)

display(ta.head())
display(ret.head())
display(link.head())

ta shape  : (1547, 37)
ret shape : (102309, 11)
link shape: (102, 7)


,company,ticker,date,file,total_words,pres_words,qa_words,core_hits_total,core_hits_pres,core_hits_qa,adj_hits_total,adj_hits_pres,adj_hits_qa,lm_positive_total,lm_negative_total,lm_uncertainty_count_total,lm_tone_total,lm_uncertainty_total,lm_positive_pres,lm_negative_pres,lm_uncertainty_count_pres,lm_tone_pres,lm_uncertainty_pres,lm_positive_qa,lm_negative_qa,lm_uncertainty_count_qa,lm_tone_qa,lm_uncertainty_qa,fiscal_quarter,fiscal_year,fiscal_period,core_per_1000_total,core_per_1000_pres,core_per_1000_qa,adj_per_1000_total,adj_per_1000_pres,adj_per_1000_qa
0,3M Co,MMM,2022-04-26,2022-Apr-26-MMM.N-140622159458-Transcript.txt,11615,5274,6153,0,0,0,9,9,0,134,174,111,-0.129870,0.010285,88,79,40,0.053892,0.008183,46,94,71,-0.342857,0.012354,1,2022,Q1 2022,0.0,0.0,0.0,0.774860,1.706485,0.000000
1,3M Co,MMM,2022-01-25,2022-Jan-25-MMM.N-140663487123-Transcript.txt,11937,4250,7495,0,0,0,9,4,5,194,143,93,0.151335,0.008356,99,59,23,0.253165,0.005873,95,83,70,0.067416,0.009925,4,2021,Q4 2021,0.0,0.0,0.0,0.753958,0.941176,0.667111
2,3M Co,MMM,2022-07-26,2022-Jul-26-MMM.N-140270570993-Transcript.txt,13203,5024,7976,0,0,0,8,7,1,226,171,137,0.138539,0.011056,143,65,50,0.375000,0.010755,83,105,87,-0.117021,0.011493,2,2022,Q2 2022,0.0,0.0,0.0,0.605923,1.393312,0.125376
3,3M Co,MMM,2022-10-25,2022-Oct-25-MMM.N-140586063160-Transcript.txt,10815,4276,6369,0,0,0,25,7,18,168,114,103,0.191489,0.010216,78,53,44,0.190840,0.011253,90,60,59,0.200000,0.009781,3,2022,Q3 2022,0.0,0.0,0.0,2.311604,1.637044,2.826189
4,3M Co,MMM,2023-04-25,2023-Apr-25-MMM.N-136949071306-Transcript.txt,10820,3806,6832,0,0,0,9,6,3,166,117,92,0.173145,0.009065,68,55,33,0.105691,0.009346,98,61,59,0.232704,0.009125,1,2023,Q1 2023,0.0,0.0,0.0,0.831793,1.576458,0.439110


,PERMNO,HdrCUSIP,Ticker,PERMCO,IssuerNm,DlyCalDt,DlyPrc,DlyRet,DlyVol,vwretd,sprtrn
0,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-03,87.90,0.007912,10653484,0.006155,0.006374
1,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-04,88.84,0.010694,11958975,-0.002351,-0.000630
2,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-05,86.46,-0.026790,11236680,-0.021879,-0.019393
3,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-06,86.34,0.002313,7918392,0.000161,-0.000964
4,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-07,87.51,0.013551,9836795,-0.004164,-0.004050


,PERMNO,HdrCUSIP,CUSIP,Ticker,PERMCO,IssuerNm,DlyCalDt
0,10104,68389X10,68389X10,ORCL,8045,ORACLE CORP,2025-12-31
1,10107,59491810,59491810,MSFT,8048,MICROSOFT CORP,2025-12-31
2,10145,43851610,43851610,HON,22168,HONEYWELL INTERNATIONAL INC,2025-12-31
3,11308,19121610,19121610,KO,20468,COCA COLA CO,2025-12-31
4,11850,30231G10,30231G10,XOM,20678,EXXON MOBIL CORP,2025-12-31


## 3. Basic cleaning

In [4]:
# Dates
ta["date"] = pd.to_datetime(ta["date"], errors="coerce")
ret["DlyCalDt"] = pd.to_datetime(ret["DlyCalDt"], errors="coerce")

# Convert comma-decimal columns in textual_analysis_results.csv to numeric
comma_decimal_cols = [
    "lm_tone_total", "lm_uncertainty_total",
    "lm_tone_pres", "lm_uncertainty_pres",
    "lm_tone_qa", "lm_uncertainty_qa",
    "core_per_1000_total", "core_per_1000_pres", "core_per_1000_qa",
    "adj_per_1000_total", "adj_per_1000_pres", "adj_per_1000_qa",
]

for col in comma_decimal_cols:
    if col in ta.columns:
        ta[col] = (
            ta[col]
            .astype(str)
            .str.replace(",", ".", regex=False)
            .replace("nan", np.nan)
            .astype(float)
        )

# Ticker cleanup
ta["ticker"] = ta["ticker"].astype(str).str.upper().str.strip()
link["Ticker"] = link["Ticker"].astype(str).str.upper().str.strip()

# PERMNO / returns cleanup
ret["PERMNO"] = pd.to_numeric(ret["PERMNO"], errors="coerce").astype("Int64")
link["PERMNO"] = pd.to_numeric(link["PERMNO"], errors="coerce").astype("Int64")
ret["DlyRet"] = pd.to_numeric(ret["DlyRet"], errors="coerce")
ret["vwretd"] = pd.to_numeric(ret["vwretd"], errors="coerce")

print(ta.dtypes.head(20))

company                               object
ticker                                object
date                          datetime64[ns]
file                                  object
total_words                            int64
pres_words                             int64
qa_words                               int64
core_hits_total                        int64
core_hits_pres                         int64
core_hits_qa                           int64
adj_hits_total                         int64
adj_hits_pres                          int64
adj_hits_qa                            int64
lm_positive_total                      int64
lm_negative_total                      int64
lm_uncertainty_count_total             int64
lm_tone_total                        float64
lm_uncertainty_total                 float64
lm_positive_pres                       int64
lm_negative_pres                       int64
dtype: object


## 4. Linking Table

In [5]:
# Keep one row per ticker/PERMNO pair in link table
link_unique = (
    link[["PERMNO", "Ticker", "IssuerNm"]]
    .drop_duplicates()
    .copy()
)

display(link_unique.head())

,PERMNO,Ticker,IssuerNm
0,10104,ORCL,ORACLE CORP
1,10107,MSFT,MICROSOFT CORP
2,10145,HON,HONEYWELL INTERNATIONAL INC
3,11308,KO,COCA COLA CO
4,11850,XOM,EXXON MOBIL CORP


## 5. Link earnings calls to PERMNO

In [6]:
ta_linked = ta.merge(
    link_unique,
    left_on="ticker",
    right_on="Ticker",
    how="left"
)

print("Rows without PERMNO after ticker merge:", ta_linked["PERMNO"].isna().sum())
display(
    ta_linked[ta_linked["PERMNO"].isna()][["company", "ticker", "file"]].drop_duplicates()
)
display(ta_linked.head())

Rows without PERMNO after ticker merge: 0


,company,ticker,file


,company,ticker,date,file,total_words,pres_words,qa_words,core_hits_total,core_hits_pres,core_hits_qa,adj_hits_total,adj_hits_pres,adj_hits_qa,lm_positive_total,lm_negative_total,lm_uncertainty_count_total,lm_tone_total,lm_uncertainty_total,lm_positive_pres,lm_negative_pres,lm_uncertainty_count_pres,lm_tone_pres,lm_uncertainty_pres,lm_positive_qa,lm_negative_qa,lm_uncertainty_count_qa,lm_tone_qa,lm_uncertainty_qa,fiscal_quarter,fiscal_year,fiscal_period,core_per_1000_total,core_per_1000_pres,core_per_1000_qa,adj_per_1000_total,adj_per_1000_pres,adj_per_1000_qa,PERMNO,Ticker,IssuerNm
0,3M Co,MMM,2022-04-26,2022-Apr-26-MMM.N-140622159458-Transcript.txt,11615,5274,6153,0,0,0,9,9,0,134,174,111,-0.129870,0.010285,88,79,40,0.053892,0.008183,46,94,71,-0.342857,0.012354,1,2022,Q1 2022,0.0,0.0,0.0,0.774860,1.706485,0.000000,22592,MMM,3M CO
1,3M Co,MMM,2022-01-25,2022-Jan-25-MMM.N-140663487123-Transcript.txt,11937,4250,7495,0,0,0,9,4,5,194,143,93,0.151335,0.008356,99,59,23,0.253165,0.005873,95,83,70,0.067416,0.009925,4,2021,Q4 2021,0.0,0.0,0.0,0.753958,0.941176,0.667111,22592,MMM,3M CO
2,3M Co,MMM,2022-07-26,2022-Jul-26-MMM.N-140270570993-Transcript.txt,13203,5024,7976,0,0,0,8,7,1,226,171,137,0.138539,0.011056,143,65,50,0.375000,0.010755,83,105,87,-0.117021,0.011493,2,2022,Q2 2022,0.0,0.0,0.0,0.605923,1.393312,0.125376,22592,MMM,3M CO
3,3M Co,MMM,2022-10-25,2022-Oct-25-MMM.N-140586063160-Transcript.txt,10815,4276,6369,0,0,0,25,7,18,168,114,103,0.191489,0.010216,78,53,44,0.190840,0.011253,90,60,59,0.200000,0.009781,3,2022,Q3 2022,0.0,0.0,0.0,2.311604,1.637044,2.826189,22592,MMM,3M CO
4,3M Co,MMM,2023-04-25,2023-Apr-25-MMM.N-136949071306-Transcript.txt,10820,3806,6832,0,0,0,9,6,3,166,117,92,0.173145,0.009065,68,55,33,0.105691,0.009346,98,61,59,0.232704,0.009125,1,2023,Q1 2023,0.0,0.0,0.0,0.831793,1.576458,0.439110,22592,MMM,3M CO


**Sanity check to see if each ticker really only equals 1 PERMNO**

In [7]:
print("Unique tickers in textual results:", ta["ticker"].nunique())
print("Unique tickers matched to PERMNO:", ta_linked.loc[ta_linked["PERMNO"].notna(), "ticker"].nunique())

display(
    ta_linked.groupby("ticker", dropna=False)["PERMNO"]
    .nunique()
    .sort_values(ascending=False)
    .head(20)
)

Unique tickers in textual results: 98
Unique tickers matched to PERMNO: 98


ticker
AAPL    1
PG      1
PEP     1
ORCL    1
NVDA    1
NOW     1
NKE     1
NFLX    1
NEE     1
MSFT    1
MS      1
MRK     1
MO      1
MMM     1
META    1
MET     1
MDT     1
MDLZ    1
MCD     1
MA      1
Name: PERMNO, dtype: int64

## 6. Compute daily abnormal return

Using your chosen market-adjusted specification:

`AR_{i,t} = DlyRet_{i,t} - vwretd_t`

In [8]:
ret = ret.copy()
ret["abret"] = ret["DlyRet"] - ret["vwretd"]

# Keep only what is needed
ret_small = (
    ret[["PERMNO", "Ticker", "IssuerNm", "DlyCalDt", "DlyRet", "vwretd", "abret"]]
    .sort_values(["PERMNO", "DlyCalDt"])
    .reset_index(drop=True)
)

display(ret_small.head())

,PERMNO,Ticker,IssuerNm,DlyCalDt,DlyRet,vwretd,abret
0,10104,ORCL,ORACLE CORP,2022-01-03,0.007912,0.006155,0.001757
1,10104,ORCL,ORACLE CORP,2022-01-04,0.010694,-0.002351,0.013045
2,10104,ORCL,ORACLE CORP,2022-01-05,-0.026790,-0.021879,-0.004911
3,10104,ORCL,ORACLE CORP,2022-01-06,0.002313,0.000161,0.002152
4,10104,ORCL,ORACLE CORP,2022-01-07,0.013551,-0.004164,0.017715


## 7. Prepare event table

Sort each firm's earnings calls. The `next_call_date` column is retained for reference only; the long-run return now uses a fixed CAR[+2,+30] window and does not depend on it.

In [9]:
events = ta_linked.copy()
events = events.sort_values(["PERMNO", "date"]).reset_index(drop=True)
events["next_call_date"] = events.groupby("PERMNO")["date"].shift(-1)

display(events[["company", "ticker", "date", "PERMNO", "next_call_date"]].head(10))

,company,ticker,date,PERMNO,next_call_date
0,Oracle Corp,ORCL,2022-03-10,10104,2022-06-13
1,Oracle Corp,ORCL,2022-06-13,10104,2022-09-12
2,Oracle Corp,ORCL,2022-09-12,10104,2022-12-12
3,Oracle Corp,ORCL,2022-12-12,10104,2023-03-09
4,Oracle Corp,ORCL,2023-03-09,10104,2023-06-12
5,Oracle Corp,ORCL,2023-06-12,10104,2023-09-11
6,Oracle Corp,ORCL,2023-09-11,10104,2023-12-11
7,Oracle Corp,ORCL,2023-12-11,10104,2024-03-11
8,Oracle Corp,ORCL,2024-03-11,10104,2024-06-11
9,Oracle Corp,ORCL,2024-06-11,10104,2024-09-09


## 8. Helper functions for event-window extraction

Two important choices here:

- `CAR[-1,+1]` uses **trading days**, not calendar days.
- `CAR[+2,+30]` is the medium-run window: it starts 2 trading days after the event (avoiding overlap with the short-run window) and ends 30 trading days after the event (~6 calendar weeks), well before the next quarterly call and free of next-quarter contamination.
- If the earnings call date itself is not a trading day, the notebook shifts to the **next available trading day** for that firm.

In [10]:
def get_trading_dates_for_firm(df_firm):
    return df_firm["DlyCalDt"].dropna().drop_duplicates().sort_values().tolist()

def get_event_trading_day(df_firm, event_date):
    # first trading day on or after the event date
    dates = df_firm.loc[df_firm["DlyCalDt"] >= event_date, "DlyCalDt"].drop_duplicates().sort_values()
    if len(dates) == 0:
        return pd.NaT
    return dates.iloc[0]

def compute_car_minus1_plus1(df_firm, event_date):
    event_trading_day = get_event_trading_day(df_firm, event_date)
    if pd.isna(event_trading_day):
        return pd.Series({
            "event_trading_day": pd.NaT,
            "car_m1_p1": np.nan,
            "n_days_car": 0
        })

    firm_dates = df_firm["DlyCalDt"].drop_duplicates().sort_values().reset_index(drop=True)
    try:
        idx = firm_dates[firm_dates == event_trading_day].index[0]
    except IndexError:
        return pd.Series({
            "event_trading_day": pd.NaT,
            "car_m1_p1": np.nan,
            "n_days_car": 0
        })

    start_idx = max(idx - 1, 0)
    end_idx = min(idx + 1, len(firm_dates) - 1)
    window_dates = firm_dates.iloc[start_idx:end_idx + 1]

    temp = df_firm[df_firm["DlyCalDt"].isin(window_dates)].copy()

    return pd.Series({
        "event_trading_day": event_trading_day,
        "car_m1_p1": temp["abret"].sum(skipna=True),
        "n_days_car": temp["abret"].notna().sum()
    })
# Fixed window offsets for the medium-run return (trading days relative to event day 0)
LONG_RUN_START_OFFSET = 2   # first day counted: +2 (skips the [-1,+1] announcement window)
LONG_RUN_END_OFFSET   = 30  # last  day counted: +30 (~6 calendar weeks; well clear of the next call)

def compute_long_run_abret(df_firm, event_date):
    """CAR[+2, +30]: cumulative abnormal return from trading day +2 to +30
    relative to the event trading day (day 0).

    Rationale:
    - Starts at +2 so it does not overlap with the short-run CAR[-1,+1] window.
    - Ends at +30 (~6 calendar weeks), well before the next quarterly call
      (~63 trading days away), eliminating next-quarter contamination.
    - Window length is fixed, so observations are fully comparable across firms
      and quarters regardless of when the next call occurs.
    """
    event_trading_day = get_event_trading_day(df_firm, event_date)
    if pd.isna(event_trading_day):
        return pd.Series({
            "long_run_start": pd.NaT,
            "long_run_end": pd.NaT,
            "long_run_abret": np.nan,
            "n_days_long_run": 0
        })

    firm_dates = df_firm["DlyCalDt"].drop_duplicates().sort_values().reset_index(drop=True)
    try:
        event_idx = firm_dates[firm_dates == event_trading_day].index[0]
    except IndexError:
        return pd.Series({
            "long_run_start": pd.NaT,
            "long_run_end": pd.NaT,
            "long_run_abret": np.nan,
            "n_days_long_run": 0
        })

    start_idx = event_idx + LONG_RUN_START_OFFSET
    end_idx   = event_idx + LONG_RUN_END_OFFSET

    if start_idx >= len(firm_dates):
        return pd.Series({
            "long_run_start": pd.NaT,
            "long_run_end": pd.NaT,
            "long_run_abret": np.nan,
            "n_days_long_run": 0
        })

    # Clip to available data (edge case: last observation in the sample)
    end_idx = min(end_idx, len(firm_dates) - 1)

    window_dates   = firm_dates.iloc[start_idx:end_idx + 1]
    long_run_start = window_dates.iloc[0]
    long_run_end   = window_dates.iloc[-1]

    temp = df_firm[df_firm["DlyCalDt"].isin(window_dates)].copy()

    return pd.Series({
        "long_run_start":  long_run_start,
        "long_run_end":    long_run_end,
        "long_run_abret":  temp["abret"].sum(skipna=True),
        "n_days_long_run": temp["abret"].notna().sum()
    })


## 9. Run the extraction firm by firm

In [11]:
results = []

for _, row in events.iterrows():
    permno = row["PERMNO"]
    event_date = row["date"]
    next_call_date = row["next_call_date"]

    if pd.isna(permno) or pd.isna(event_date):
        out = row.to_dict()
        out.update({
            "event_trading_day": pd.NaT,
            "car_m1_p1": np.nan,
            "n_days_car": 0,
            "long_run_start": pd.NaT,
            "long_run_end": pd.NaT,
            "long_run_abret": np.nan,
            "n_days_long_run": 0
        })
        results.append(out)
        continue

    df_firm = ret_small[ret_small["PERMNO"] == permno].sort_values("DlyCalDt").copy()

    car_info = compute_car_minus1_plus1(df_firm, event_date)
    long_info = compute_long_run_abret(df_firm, event_date)

    out = row.to_dict()
    out.update(car_info.to_dict())
    out.update(long_info.to_dict())
    results.append(out)

financial_event_df = pd.DataFrame(results)

display(
    financial_event_df[
        ["company", "date", "PERMNO", "event_trading_day", "car_m1_p1", "n_days_car", "long_run_start", "long_run_end", "long_run_abret", "n_days_long_run"]
    ].head(15)
)

,company,date,PERMNO,event_trading_day,car_m1_p1,n_days_car,long_run_start,long_run_end,long_run_abret,n_days_long_run
0,Oracle Corp,2022-03-10,10104,2022-03-10,0.051903,3,2022-03-14,2022-04-22,-0.026852,29
1,Oracle Corp,2022-06-13,10104,2022-06-13,0.100445,3,2022-06-15,2022-07-27,0.004816,29
2,Oracle Corp,2022-09-12,10104,2022-09-12,0.031875,3,2022-09-14,2022-10-24,0.010884,29
3,Oracle Corp,2022-12-12,10104,2022-12-12,-0.006554,3,2022-12-14,2023-01-26,0.092298,29
4,Oracle Corp,2023-03-09,10104,2023-03-09,-0.014579,3,2023-03-13,2023-04-21,0.068929,29
5,Oracle Corp,2023-06-12,10104,2023-06-12,0.067970,3,2023-06-14,2023-07-26,-0.048114,29
6,Oracle Corp,2023-09-11,10104,2023-09-11,-0.124762,3,2023-09-13,2023-10-23,0.013064,29
7,Oracle Corp,2023-12-11,10104,2023-12-11,-0.115474,3,2023-12-13,2024-01-25,0.086744,29
8,Oracle Corp,2024-03-11,10104,2024-03-11,0.111943,3,2024-03-13,2024-04-23,-0.079175,29
9,Oracle Corp,2024-06-11,10104,2024-06-11,0.103497,3,2024-06-13,2024-07-25,-0.016676,29


## 10. Quick quality checks

In [12]:
print("Missing PERMNO rows:", financial_event_df["PERMNO"].isna().sum())
print("Missing CAR rows   :", financial_event_df["car_m1_p1"].isna().sum())
print("Missing long-run rows:", financial_event_df["long_run_abret"].isna().sum())

print("\nDistribution of n_days_car")
print(financial_event_df["n_days_car"].value_counts(dropna=False).sort_index())

print("\nDistribution of n_days_long_run")
print(financial_event_df["n_days_long_run"].value_counts(dropna=False).sort_index().head(20))

display(financial_event_df[financial_event_df["n_days_car"] < 2][["company", "date", "PERMNO", "event_trading_day", "n_days_car"]].head(20))

Missing PERMNO rows: 0
Missing CAR rows   : 0
Missing long-run rows: 0

Distribution of n_days_car
n_days_car
3    1547
Name: count, dtype: int64

Distribution of n_days_long_run
n_days_long_run
7        3
12       2
13       2
18       1
22       1
26       2
27       3
28       2
29    1528
30       3
Name: count, dtype: int64


,company,date,PERMNO,event_trading_day,n_days_car


For `CAR[-1,+1]`, you will usually expect **3 trading days** when the firm has full return coverage for the window.  
For `CAR[+2,+30]`, you will usually expect **29 trading days** (days +2 through +30 inclusive).  
If some rows show fewer days, inspect them manually (typically only the last observation in the sample where data ends early).

## 11. Create final regression-ready dataset

In [13]:
final_cols_front = [
    "company", "ticker", "date", "file", "PERMNO", "Ticker", "IssuerNm",
    "event_trading_day", "next_call_date",
    "car_m1_p1", "n_days_car",
    "long_run_start", "long_run_end", "long_run_abret", "n_days_long_run"
]

remaining_cols = [c for c in financial_event_df.columns if c not in final_cols_front]
final_df = financial_event_df[final_cols_front + remaining_cols].copy()

display(final_df.head())
print(final_df.shape)

,company,ticker,date,file,PERMNO,Ticker,IssuerNm,event_trading_day,next_call_date,car_m1_p1,n_days_car,long_run_start,long_run_end,long_run_abret,n_days_long_run,total_words,pres_words,qa_words,core_hits_total,core_hits_pres,core_hits_qa,adj_hits_total,adj_hits_pres,adj_hits_qa,lm_positive_total,lm_negative_total,lm_uncertainty_count_total,lm_tone_total,lm_uncertainty_total,lm_positive_pres,lm_negative_pres,lm_uncertainty_count_pres,lm_tone_pres,lm_uncertainty_pres,lm_positive_qa,lm_negative_qa,lm_uncertainty_count_qa,lm_tone_qa,lm_uncertainty_qa,fiscal_quarter,fiscal_year,fiscal_period,core_per_1000_total,core_per_1000_pres,core_per_1000_qa,adj_per_1000_total,adj_per_1000_pres,adj_per_1000_qa
0,Oracle Corp,ORCL,2022-03-10,2022-Mar-10-ORCL.N-139346710200-Transcript.txt,10104,ORCL,ORACLE CORP,2022-03-10,2022-06-13,0.051903,3,2022-03-14,2022-04-22,-0.026852,29,7503,4184,3197,0,0,0,1,1,0,103,41,47,0.430556,0.006636,76,19,23,0.600000,0.005884,27,21,24,0.125000,0.007807,3,2022,Q3 2022,0.000000,0.000000,0.000000,0.133280,0.239006,0.000000
1,Oracle Corp,ORCL,2022-06-13,2022-Jun-13-ORCL.N-140185486493-Transcript.txt,10104,ORCL,ORACLE CORP,2022-06-13,2022-09-12,0.100445,3,2022-06-15,2022-07-27,0.004816,29,6921,3452,3333,1,1,0,3,3,0,79,39,45,0.338983,0.006904,42,12,22,0.555556,0.006875,37,26,23,0.174603,0.007174,4,2022,Q4 2022,0.144488,0.289687,0.000000,0.433463,0.869061,0.000000
2,Oracle Corp,ORCL,2022-09-12,2022-Sep-12-ORCL.N-138707918626-Transcript.txt,10104,ORCL,ORACLE CORP,2022-09-12,2022-12-12,0.031875,3,2022-09-14,2022-10-24,0.010884,29,6634,3055,3448,7,7,0,1,0,1,59,47,52,0.113208,0.008321,23,11,23,0.352941,0.008200,36,35,29,0.014085,0.008690,1,2023,Q1 2023,1.055170,2.291326,0.000000,0.150739,0.000000,0.290023
3,Oracle Corp,ORCL,2022-12-12,2022-Dec-12-ORCL.N-140500431181-Transcript.txt,10104,ORCL,ORACLE CORP,2022-12-12,2023-03-09,-0.006554,3,2022-12-14,2023-01-26,0.092298,29,7684,4586,2974,13,8,5,8,5,3,68,42,43,0.236364,0.005929,40,9,20,0.632653,0.004662,28,32,23,-0.066667,0.008036,2,2023,Q2 2023,1.691827,1.744440,1.681237,1.041124,1.090275,1.008742
4,Oracle Corp,ORCL,2023-03-09,2023-Mar-09-ORCL.N-138300942700-Transcript.txt,10104,ORCL,ORACLE CORP,2023-03-09,2023-06-12,-0.014579,3,2023-03-13,2023-04-21,0.068929,29,7267,3626,3515,45,8,37,1,0,1,72,39,45,0.297297,0.006555,38,12,20,0.520000,0.005981,34,26,25,0.133333,0.007316,3,2023,Q3 2023,6.192376,2.206288,10.526316,0.137608,0.000000,0.284495


(1547, 48)


## 12. Export outputs

In [14]:
# output directory
OUTPUT_DIR = Path("../../data/processed")

OUTPUT_EVENTS = OUTPUT_DIR / "financial_event_dataset.csv"
OUTPUT_RETURNS = OUTPUT_DIR / "daily_returns_with_abret.csv"

final_df.to_csv(OUTPUT_EVENTS, index=False)
ret_small.to_csv(OUTPUT_RETURNS, index=False)

print("Saved:", OUTPUT_EVENTS.resolve())
print("Saved:", OUTPUT_RETURNS.resolve())

Saved: /Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/processed/financial_event_dataset.csv
Saved: /Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/processed/daily_returns_with_abret.csv


## 13. Optional: very small diagnostic sample

Use this to inspect one firm manually and verify the event windows.

In [15]:
sample_firm = final_df["company"].dropna().iloc[0]
sample_permno = final_df.loc[final_df["company"] == sample_firm, "PERMNO"].iloc[0]

print("Sample firm :", sample_firm)
print("Sample PERMNO:", sample_permno)

display(final_df[final_df["company"] == sample_firm][[
    "company", "date", "event_trading_day", "car_m1_p1",
    "long_run_start", "long_run_end", "long_run_abret"
]].sort_values("date"))

Sample firm : Oracle Corp
Sample PERMNO: 10104


,company,date,event_trading_day,car_m1_p1,long_run_start,long_run_end,long_run_abret
0,Oracle Corp,2022-03-10,2022-03-10,0.051903,2022-03-14,2022-04-22,-0.026852
1,Oracle Corp,2022-06-13,2022-06-13,0.100445,2022-06-15,2022-07-27,0.004816
2,Oracle Corp,2022-09-12,2022-09-12,0.031875,2022-09-14,2022-10-24,0.010884
3,Oracle Corp,2022-12-12,2022-12-12,-0.006554,2022-12-14,2023-01-26,0.092298
4,Oracle Corp,2023-03-09,2023-03-09,-0.014579,2023-03-13,2023-04-21,0.068929
5,Oracle Corp,2023-06-12,2023-06-12,0.067970,2023-06-14,2023-07-26,-0.048114
6,Oracle Corp,2023-09-11,2023-09-11,-0.124762,2023-09-13,2023-10-23,0.013064
7,Oracle Corp,2023-12-11,2023-12-11,-0.115474,2023-12-13,2024-01-25,0.086744
8,Oracle Corp,2024-03-11,2024-03-11,0.111943,2024-03-13,2024-04-23,-0.079175
9,Oracle Corp,2024-06-11,2024-06-11,0.103497,2024-06-13,2024-07-25,-0.016676
